In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from glob import glob
from autopath.ap_PLIP import ProteinLigandAnalyzer
import os
import numpy as np
from scipy.stats import linregress, pearsonr, spearmanr, kendalltau
from sklearn.metrics import r2_score

os.makedirs('plots_mmgbsa', exist_ok=True)

Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.

****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other     *
* Uses of JAX in the same code.          *
******************************************

Please install openmmtools to use Girsanov reweighting.
/gpfs/home/mllanos/micromamba/envs/autopath/lib/python3.12/site-packages/MDAnalysis/topology/tables.py:52: DeprecationWarning: Deprecat

In [4]:
sysname = '3ptb'
allsystems = glob(f'{sysname}/*/FINAL_RESULTS_mmpbsa.dat')
print(f'Found {len(allsystems)} systems with MMPBSA results.')
alldfs = []
for pathfname in allsystems:
    expname = pathfname.split('/')[1]
    df_diff = ProteinLigandAnalyzer.parse_mmpbsa_differences_table(pathfname)
    df_diff['system'] = sysname
    df_diff['experiment'] = expname
    alldfs.append(df_diff)
    
df_all = pd.concat(alldfs, ignore_index=True)
df_all = df_all[~df_all['Component'].isin(['ENPOLAR', 'EDISPER', 'DELTA G gas', 'DELTA G solv'])]
df_all.head()

Found 2 systems with MMPBSA results.


,Component,Average,Std_Dev,Std_Err_Mean,system,experiment
0,VDWAALS,-21.5395,2.4457,0.1641,3ptb,mmgbsa_igb8_WAT
1,EEL,7.9429,8.3584,0.5610,3ptb,mmgbsa_igb8_WAT
2,EGB,-10.3730,6.9007,0.4631,3ptb,mmgbsa_igb8_WAT
3,ESURF,-2.7472,0.0864,0.0058,3ptb,mmgbsa_igb8_WAT
6,DELTA TOTAL,-26.7169,3.0171,0.2025,3ptb,mmgbsa_igb8_WAT


In [5]:
experiment_mapping = {
    'mmgbsa_WAT_ALL_igb5': 'MMGBSA (OBC2) - WET',
    'mmgbsa_WAT_ALL_igb8': 'MMGBSA (GBn2) - WET',
    'mmgbsa_DRY_ALL_igb5': 'MMGBSA (OBC2) - DRY',
    'mmgbsa_DRY_ALL_igb8': 'MMGBSA (GBn2) - DRY'
}